# Lab 03 · Store, retrieve and inspect ChromaDB

Module D • **Requires `chromadb` installation** (no cloud account). Uses precomputed toy vectors, so Chroma makes no model/API call.

**Workflow:** read → predict → run → edit the marked variable → rerun → answer the linked questions in `assessments/quiz.html`.  \n**No answer needs a screenshot.** Keep your notebook output visible while taking the quiz.

# Your task board · Run first, answer later

Use the teaching cells below to determine the answers. For an **EDIT** task, change the marked variable and rerun the specified cell. The hints and full answers are intentionally **open at the end of this notebook**. Try the questions first.

**D1 · NAT · after cell 03 · D2** — After adding all toy items, what is collection.count()?  
NAT: `_________`

**D2 · NAT · after cell 03 · D3** — With K=3, how many IDs are in the first query result?  
NAT: `_________`

**D3 · NAT · after cell 03 · D3 EDIT** — After setting K=5, how many IDs are returned?  
NAT: `_________`

**D4 · NAT · after cell 03 · D4** — What is squared L2 distance from Cat to Cat?  
NAT: `_________`

**D5 · NAT · after cell 03 · D4** — What is the manually computed Cat–Dog squared L2? Give 4 decimal places.  
NAT: `_________`

**D6 · MCQ · after cell 03 · D3** — For Chroma’s local default l2 metric, what does a *smaller* returned distance mean?
  - **A.** Less similar
  - **B.** Closer
  - **C.** Higher dot product necessarily
  - **D.** More tokens

**D7 · MCQ · after cell 03 · D5** — What does the `where={"kind":"text"}` argument do?
  - **A.** Retrains the model
  - **B.** Applies a metadata filter
  - **C.** Switches HNSW to IVF
  - **D.** Changes vector dimension

**D8 · MCQ · after cell 03 · D6** — Which constructor persists local Chroma data to disk?
  - **A.** chromadb.Client(Settings(persist_directory=...))
  - **B.** chromadb.PersistentClient(path=...)
  - **C.** PCA(n_components=3)
  - **D.** KMeans(n_clusters=3)

---

**Assessment:** [Interactive self-check](../assessments/quiz.html?module=D) · [Full question sheet](../assessments/QUESTION_SHEET.md). You can learn without grading; 70% unlocks the next quiz on the portal.

**Official documentation:** [Chroma collections](https://cookbook.chromadb.dev/core/collections/) · [Chroma HNSW config](https://cookbook.chromadb.dev/core/configuration/) · [Chroma storage](https://cookbook.chromadb.dev/core/storage-layout/).

In [ ]:
from pathlib import Path
import numpy as np
from IPython.display import display

def find_workshop_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/embedding_workshop')]
    for p in candidates:
        if (p / 'data' / 'field_notes.md').is_file():
            return p
    raise FileNotFoundError('Cannot find data/field_notes.md. Unzip the complete workshop, not just one notebook. Check README.md.')
ROOT = find_workshop_root()
print('Workshop root:', ROOT)


## D0 · Install once
Run the next cell. On managed notebooks, restart the kernel if prompted after installation. Modern Chroma uses `PersistentClient(path=...)`, not `Client(Settings(persist_directory=...))`.

In [ ]:
%pip -q install 'chromadb>=1.0,<3.0' 

## D1 · Initialize the persistent Chroma collection
`configuration={"hnsw":{"space":"l2"}}` specifies the single-node HNSW index and **squared** Euclidean distance. It is also the usual default on local Chroma. Explicit is better for teaching.

In [ ]:
import json,chromadb
items=json.loads((ROOT/'data/toy_vectors.json').read_text())
client=chromadb.PersistentClient(path=str(ROOT/'data'/'chroma_student_db'))
NAME='student_vectors_l2'
# Repeatability: delete and recreate this TEACHING collection on every full run.
existing_names=[c.name for c in client.list_collections()]
if NAME in existing_names:
    client.delete_collection(NAME)
collection=client.create_collection(name=NAME,configuration={'hnsw':{'space':'l2'}},embedding_function=None)
print('collection =',collection.name)
print('persistent_path =',ROOT/'data'/'chroma_student_db')

## D2 · Explicitly insert vectors, documents and metadata
With `embeddings=` supplied, Chroma need not embed them. Each `id` is unique; metadata supports filters.

In [ ]:
collection.add(ids=[it['id'] for it in items],
               documents=[it['text'] for it in items],
               embeddings=[it['vector'] for it in items],
               metadatas=[{'kind':it['kind'],'label':it['label']} for it in items])
print('count =',collection.count())
print('stored_ids =',sorted(collection.get()['ids']))

## D3 · Query a vector; inspect scores
✏️ `K = 3` first. Query **Cat** using the same 6D toy space. The input itself will usually appear first at distance 0.

In [ ]:
query_vector=next(it['vector'] for it in items if it['id']=='cat')
K = 3 # ✏️ Change to 5, rerun, then restore to 3.
response=collection.query(query_embeddings=[query_vector],n_results=K,
                          include=['documents','distances','metadatas'])
for identifier,doc,dist in zip(response['ids'][0],response['documents'][0],response['distances'][0]):
    print(f'{identifier:15s} squared_L2={dist:.6f} | {doc}')
print('result_count =',len(response['ids'][0]))

## D4 · Test an exact formula independently

In [ ]:
x=np.array(next(it['vector'] for it in items if it['id']=='cat'))
y=np.array(next(it['vector'] for it in items if it['id']=='dog'))
print('manual_squared_l2_cat_dog =',round(float(np.sum((x-y)**2)),7))
print('manual_squared_l2_cat_cat =',round(float(np.sum((x-x)**2)),7))

## D5 · Filter by metadata: text only
This is a **structured filter** combined with vector retrieval, not a different embedding algorithm.

In [ ]:
filtered=collection.query(query_embeddings=[query_vector],n_results=5,
                          where={'kind':'text'},include=['documents','distances','metadatas'])
print('kinds =',[m['kind'] for m in filtered['metadatas'][0]])
print('ids =',filtered['ids'][0])
print('returned =',len(filtered['ids'][0]))

## D6 · Verify persistence
Re-opening the same path gives the stored collection. It is **not** necessary to call `.persist()`.

In [ ]:
again=chromadb.PersistentClient(path=str(ROOT/'data'/'chroma_student_db'))
reopened=again.get_collection(NAME,embedding_function=None)
print('reopened_count =',reopened.count())
print('reopened_same_ids =',set(reopened.get()['ids'])==set(collection.get()['ids']))

### Finish Module D
Answer **D1–D8**. For the project challenge, try `configuration={"hnsw":{"space":"cosine"}}` in a **different collection name**; Chroma reports **cosine distance**, not cosine similarity.

## Hints · Use these only after attempting the tasks

- **D1:** Read `collection.count()` after insertion.
- **D2:** Look at the length of `response["ids"][0]`.
- **D3:** Change K then rerun D3 and count returned IDs.
- **D4:** Subtract a vector from itself.
- **D5:** Use `np.sum((x-y)**2)` in D4.
- **D6:** Think minimum distance, not maximum similarity.
- **D7:** Read `where` as a filter on the stored metadata.
- **D8:** Inspect the client constructor used in D1 and D6.

These are learning hints, not the solutions. Scroll down for the fully open answer key.


## Open answers · Full worked key and explanations

The answers below refer to the **supplied dataset and stated default/EDIT settings**. If you changed additional inputs, your results may differ. Compare the experiment and formula—not just the number.

**D1: 10**  
Ten unique vectors added.

**D2: 3**  
There are at least three stored items.

**D3: 5**  
With n_results=5 and ten available vectors, five IDs return.

**D4: 0**  
Identical vectors have zero squared distance.

**D5: 0.0298**  
Sum of six coordinate-wise squared differences = 0.0298.

**D6: B — Closer**  
Squared L2 is a distance, so smaller is nearer.

**D7: B — Applies a metadata filter**  
It filters eligible records via metadata.

**D8: B — chromadb.PersistentClient(path=...)**  
Modern persistent local Chroma uses PersistentClient(path=...).

### Extension challenge — Interview explanation

**Challenge:** Your persistent Chroma store returns an unexpected duplicate. What would you inspect (IDs, insertion, filters, collection and distance)?

**Self-check:** Explain the trade-off aloud in 45 seconds using a concrete example.
